# 🎙️ AK-AIGC-Studio: CosyVoice 2 快速起步实验手册
> **主理人**: B1ueAkisa | **AI 总副手**: Gemini (Google DeepMind)

本 Notebook 将帮助你一键完成：
1. 挂载 Google Drive，将模型缓存和输出结果永久存放在 `AIGC_Workspace`；
2. 下载并加载阿里开源的 **CosyVoice2-0.5B** 旗舰多语言/零样本声音克隆模型；
3. 体验多情感中文、英文及方言合成，并直接试听导出！

### Step 1: 检查 GPU 状态与挂载 Google Drive

In [ ]:
# 检查显卡
!nvidia-smi

# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
WORKSPACE = '/content/drive/MyDrive/AIGC_Workspace'
MODELS_AUDIO_DIR = os.path.join(WORKSPACE, 'models/audio')
OUTPUTS_DIR = os.path.join(WORKSPACE, 'outputs')
os.makedirs(MODELS_AUDIO_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)
print(f"✅ Google Drive 工作区就绪: {WORKSPACE}")

### Step 2: 克隆 CosyVoice 官方库并安装环境依赖

In [ ]:
%cd /content
!git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git || true
%cd /content/CosyVoice
!pip install -q -r requirements.txt
!pip install -q modelscope sox

### Step 3: 下载 / 读取 CosyVoice 2 模型 (自动持久化到云盘)

In [ ]:
import sys
sys.path.append('/content/CosyVoice')
sys.path.append('/content/CosyVoice/third_party/Matcha-TTS')

from modelscope import snapshot_download

MODEL_DIR = '/content/drive/MyDrive/AIGC_Workspace/models/audio/CosyVoice2-0.5B'
print(f"📦 正在检查或从 ModelScope 下载 CosyVoice2-0.5B 到: {MODEL_DIR}")
snapshot_download('iic/CosyVoice2-0.5B', local_dir=MODEL_DIR)
print("✅ 模型权重准备就绪！")

### Step 4: 加载模型并进行语音合成测试

In [ ]:
import torchaudio
from IPython.display import Audio, display
from cosyvoice.cli.cosyvoice import CosyVoice2

print("🚀 正在初始化 CosyVoice2 模型并载入显存...")
cosyvoice = CosyVoice2(MODEL_DIR, load_jit=False, load_trt=False, fp16=True)
print("✅ 模型加载完成！")

# 示例文案
text = "欢迎来到 AK-AIGC-Studio！这里是你的专属多模态生成工坊，由主理人 B1ueAkisa 与总副手 Gemini 共同打造。"
print(f"🎙️ 开始合成: {text}")

# 使用内置音色推理
output_wav = os.path.join(OUTPUTS_DIR, 'welcome_gemini.wav')
for i, j in enumerate(cosyvoice.inference_sft(text, '中文女')):
    torchaudio.save(output_wav, j['tts_speech'], cosyvoice.sample_rate)
    break

print(f"🎉 合成完成！已自动存入 Google Drive: {output_wav}")
display(Audio(output_wav))